# FairFace x FACET-Derived Occupation Prompt Bias

This notebook runs the main FairFace CLIP association experiment using the FACET-derived broad occupation prompt set from `configs/occupations_facet_broad.yml`.

Research focus:

- Intersectional comparison: `race x gender` groups.
- Gender decomposition: male/female differences overall and within each race.
- Race decomposition: race differences overall and within each gender.

FairFace has demographic labels but no true occupation labels. Results should be interpreted as CLIP image-text association patterns, not occupation classification or FACET image evaluation.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from clip_bias.analysis import (
    add_within_image_ranks,
    aggregate_prompt_scores,
    build_similarity_table,
    occupation_group_tests,
    summarize_by_group,
)
from clip_bias.config import build_prompts, load_occupation_config
from clip_bias.data import balanced_sample, filter_adults, load_fairface_labels, resolve_dataset_path, resolve_fairface_root
from clip_bias.model import ClipEncoder, get_device

try:
    display
except NameError:
    display = print

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

PROJECT_ROOT

## 1. Parameters

In [ ]:
FAIRFACE_ROOT = resolve_fairface_root(PROJECT_ROOT)
CONFIG_PATH = PROJECT_ROOT / "configs" / "occupations_facet_broad.yml"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "fairface_facet_broad"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SPLIT = "both"
SAMPLES_PER_GROUP = 300
RANDOM_STATE = 42
MODEL_NAME = "openai/clip-vit-base-patch32"

# Keep False to reuse generated tables and embeddings if they already exist.
FORCE_RECOMPUTE = False

print("Device:", get_device())
print("FairFace:", FAIRFACE_ROOT)
print("Config:", CONFIG_PATH)
print("Output:", OUTPUT_DIR)


## 2. Load FairFace and Build Prompts

In [ ]:
labels = load_fairface_labels(FAIRFACE_ROOT, split=SPLIT)
adult_labels = filter_adults(labels)

available = adult_labels.groupby(["race", "gender"]).size().rename("available_images").reset_index()
display(available)

sampled_labels = balanced_sample(
    adult_labels,
    samples_per_group=SAMPLES_PER_GROUP,
    random_state=RANDOM_STATE,
)
sampled_labels.to_csv(TABLE_DIR / "sampled_fairface_labels.csv", index=False)

sample_counts = sampled_labels.groupby(["race", "gender", "group"]).size().rename("sampled_images").reset_index()
print("Sampled images:", len(sampled_labels))
display(sample_counts)

In [ ]:
config = load_occupation_config(CONFIG_PATH)
prompt_rows = build_prompts(config)
prompt_df = pd.DataFrame(prompt_rows)
prompt_df.to_csv(TABLE_DIR / "occupation_prompts.csv", index=False)

print("Occupations:", prompt_df["occupation"].nunique())
print("Prompt templates:", prompt_df["template"].nunique())
print("Total prompts:", len(prompt_df))
display(prompt_df.head(20))

## 3. Encode with CLIP and Build Similarity Tables

This step can take time. If the output tables already exist and `FORCE_RECOMPUTE` is `False`, the notebook loads existing tables instead of re-encoding images.

In [ ]:
scores_path = TABLE_DIR / "occupation_similarity_scores.csv"
prompt_scores_path = TABLE_DIR / "similarity_scores_prompt_level.csv"
EMBEDDING_DIR = OUTPUT_DIR / "embeddings"
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)
image_embeddings_path = EMBEDDING_DIR / "image_embeddings.npy"
text_embeddings_path = EMBEDDING_DIR / "text_embeddings.npy"

if image_embeddings_path.exists() and text_embeddings_path.exists() and not FORCE_RECOMPUTE:
    image_embeddings = np.load(image_embeddings_path)
    text_embeddings = np.load(text_embeddings_path)
    print("Loaded existing CLIP embeddings:", image_embeddings.shape, text_embeddings.shape)
else:
    encoder = ClipEncoder(model_name=MODEL_NAME)
    text_embeddings = encoder.encode_texts(prompt_df["prompt"].tolist(), batch_size=64)
    image_embeddings = encoder.encode_images(sampled_labels["image_path"].tolist(), batch_size=32)
    np.save(image_embeddings_path, image_embeddings)
    np.save(text_embeddings_path, text_embeddings)
    print("Saved CLIP embeddings:", image_embeddings.shape, text_embeddings.shape)

if scores_path.exists() and not FORCE_RECOMPUTE:
    scores = pd.read_csv(scores_path)
    prompt_scores = pd.read_csv(prompt_scores_path) if prompt_scores_path.exists() else None
    print("Loaded existing occupation-level scores:", scores.shape)
else:
    prompt_scores = build_similarity_table(sampled_labels, image_embeddings, prompt_rows, text_embeddings)
    prompt_scores.to_csv(prompt_scores_path, index=False)

    scores = aggregate_prompt_scores(prompt_scores)
    scores.to_csv(scores_path, index=False)

    print("Prompt-level rows:", prompt_scores.shape)
    print("Occupation-level rows:", scores.shape)

display(scores.head())


## 4. Intersectional Race x Gender Analysis

In [ ]:
group_summary = summarize_by_group(scores)
group_summary.to_csv(TABLE_DIR / "group_occupation_summary.csv", index=False)

group_tests = occupation_group_tests(scores)
group_tests.to_csv(TABLE_DIR / "intersectional_group_tests.csv", index=False)

display(group_summary.head(20))
display(group_tests.head(20))

In [ ]:
pivot = group_summary.pivot_table(index="group", columns="occupation", values="mean_similarity")
ordered_cols = group_tests.sort_values("bias_gap", ascending=False)["occupation"].tolist()
pivot = pivot[ordered_cols]

plt.figure(figsize=(22, 7))
sns.heatmap(pivot, cmap="vlag", center=pivot.to_numpy().mean(), linewidths=0.15)
plt.title("Mean CLIP similarity by race x gender group and FACET-derived occupation prompts")
plt.xlabel("Occupation, ordered by intersectional gap")
plt.ylabel("Race x gender group")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "intersectional_group_occupation_heatmap.png", dpi=200)
plt.show()

plt.figure(figsize=(12, 10))
plot_df = group_tests.sort_values("bias_gap", ascending=False).head(30)
sns.barplot(data=plot_df, y="occupation", x="bias_gap", color="#4C78A8")
plt.title("Largest race x gender similarity gaps")
plt.xlabel("max group mean similarity - min group mean similarity")
plt.ylabel("Occupation")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "intersectional_top_gaps.png", dpi=200)
plt.show()

## 5. Gender Decomposition

This section collapses race and compares gender overall. Positive `male_minus_female` means CLIP similarity is higher for male-labelled FairFace images; negative values mean higher similarity for female-labelled images.

In [ ]:
gender_summary = (
    scores.groupby(["occupation", "category", "gender"], as_index=False)
    .agg(mean_similarity=("similarity", "mean"), std_similarity=("similarity", "std"), n=("similarity", "size"))
)

gender_pivot = gender_summary.pivot_table(index=["occupation", "category"], columns="gender", values="mean_similarity").reset_index()
gender_pivot["male_minus_female"] = gender_pivot.get("Male") - gender_pivot.get("Female")
gender_pivot["abs_gender_gap"] = gender_pivot["male_minus_female"].abs()
gender_pivot = gender_pivot.sort_values("abs_gender_gap", ascending=False)

gender_summary.to_csv(TABLE_DIR / "gender_occupation_summary.csv", index=False)
gender_pivot.to_csv(TABLE_DIR / "gender_gap_by_occupation.csv", index=False)

display(gender_pivot.head(30))

In [ ]:
plot_df = gender_pivot.sort_values("male_minus_female").copy()
plt.figure(figsize=(12, 12))
colors = ["#D62728" if value < 0 else "#1F77B4" for value in plot_df["male_minus_female"]]
plt.barh(plot_df["occupation"], plot_df["male_minus_female"], color=colors)
plt.axvline(0, color="black", linewidth=1)
plt.title("Gender association gap by occupation")
plt.xlabel("Male mean similarity - Female mean similarity")
plt.ylabel("Occupation")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "gender_gap_by_occupation.png", dpi=200)
plt.show()

## 6. Gender Gap Within Each Race

In [ ]:
within_race_gender = (
    scores.groupby(["occupation", "race", "gender"], as_index=False)
    .agg(mean_similarity=("similarity", "mean"), n=("similarity", "size"))
)
within_race_gender_pivot = within_race_gender.pivot_table(index=["occupation", "race"], columns="gender", values="mean_similarity").reset_index()
within_race_gender_pivot["male_minus_female"] = within_race_gender_pivot.get("Male") - within_race_gender_pivot.get("Female")
within_race_gender_pivot.to_csv(TABLE_DIR / "within_race_gender_gap.csv", index=False)

top_gender_occupations = gender_pivot.head(20)["occupation"].tolist()
heat = within_race_gender_pivot[within_race_gender_pivot["occupation"].isin(top_gender_occupations)]
heat = heat.pivot_table(index="race", columns="occupation", values="male_minus_female")
heat = heat[top_gender_occupations]

plt.figure(figsize=(18, 6))
sns.heatmap(heat, cmap="vlag", center=0, linewidths=0.2)
plt.title("Gender gap within each race: Male - Female similarity")
plt.xlabel("Top occupations by overall absolute gender gap")
plt.ylabel("Race")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "within_race_gender_gap_heatmap.png", dpi=200)
plt.show()

display(within_race_gender_pivot.sort_values("male_minus_female", key=lambda s: s.abs(), ascending=False).head(30))

## 7. Race Decomposition

In [ ]:
race_summary = (
    scores.groupby(["occupation", "category", "race"], as_index=False)
    .agg(mean_similarity=("similarity", "mean"), std_similarity=("similarity", "std"), n=("similarity", "size"))
)

race_gap_rows = []
for occupation, sub in race_summary.groupby("occupation"):
    means = sub.set_index("race")["mean_similarity"]
    race_gap_rows.append(
        {
            "occupation": occupation,
            "category": sub["category"].iloc[0],
            "race_gap": means.max() - means.min(),
            "max_race": means.idxmax(),
            "min_race": means.idxmin(),
        }
    )
race_gap = pd.DataFrame(race_gap_rows).sort_values("race_gap", ascending=False)

race_summary.to_csv(TABLE_DIR / "race_occupation_summary.csv", index=False)
race_gap.to_csv(TABLE_DIR / "race_gap_by_occupation.csv", index=False)

display(race_gap.head(30))

In [ ]:
plt.figure(figsize=(12, 10))
plot_df = race_gap.head(30)
sns.barplot(data=plot_df, y="occupation", x="race_gap", color="#59A14F")
plt.title("Largest race-only similarity gaps")
plt.xlabel("max race mean similarity - min race mean similarity")
plt.ylabel("Occupation")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "race_gap_by_occupation.png", dpi=200)
plt.show()

race_heat = race_summary.pivot_table(index="race", columns="occupation", values="mean_similarity")
race_heat = race_heat[race_gap["occupation"].tolist()]
plt.figure(figsize=(22, 5))
sns.heatmap(race_heat, cmap="vlag", center=race_heat.to_numpy().mean(), linewidths=0.2)
plt.title("Mean CLIP similarity by race and occupation")
plt.xlabel("Occupation, ordered by race gap")
plt.ylabel("Race")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "race_occupation_heatmap.png", dpi=200)
plt.show()

## 8. Race Gap Within Each Gender

In [ ]:
within_gender_race_rows = []
for (occupation, gender), sub in scores.groupby(["occupation", "gender"]):
    means = sub.groupby("race")["similarity"].mean()
    within_gender_race_rows.append(
        {
            "occupation": occupation,
            "gender": gender,
            "race_gap": means.max() - means.min(),
            "max_race": means.idxmax(),
            "min_race": means.idxmin(),
        }
    )
within_gender_race = pd.DataFrame(within_gender_race_rows).sort_values("race_gap", ascending=False)
within_gender_race.to_csv(TABLE_DIR / "within_gender_race_gap.csv", index=False)

plt.figure(figsize=(12, 10))
plot_df = within_gender_race.head(30)
sns.barplot(data=plot_df, y="occupation", x="race_gap", hue="gender")
plt.title("Largest race gaps within each gender")
plt.xlabel("max race mean similarity - min race mean similarity")
plt.ylabel("Occupation")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "within_gender_race_gap_top.png", dpi=200)
plt.show()

display(within_gender_race.head(30))

## 9. Rank-Based View

Absolute cosine similarities can be hard to interpret. Ranking occupations within each image shows which prompts CLIP places relatively higher for each image.

In [ ]:
ranks = add_within_image_ranks(scores)
ranks.to_csv(TABLE_DIR / "occupation_ranks.csv", index=False)

rank_summary = (
    ranks.groupby(["occupation", "group", "race", "gender"], as_index=False)
    .agg(mean_rank=("rank_desc", "mean"), median_rank=("rank_desc", "median"), top5_rate=("rank_desc", lambda s: (s <= 5).mean()))
    .sort_values("mean_rank")
)
rank_summary.to_csv(TABLE_DIR / "group_occupation_rank_summary.csv", index=False)

top_ranked = rank_summary.groupby("group").head(8).copy()
plt.figure(figsize=(12, 12))
sns.scatterplot(data=top_ranked, x="mean_rank", y="group", hue="occupation", s=80)
plt.gca().invert_xaxis()
plt.title("Top-ranked occupations per race x gender group")
plt.xlabel("Mean within-image rank, lower is more associated")
plt.ylabel("Race x gender group")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Occupation")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "top_ranked_occupations_by_group.png", dpi=200)
plt.show()

display(rank_summary.head(40))

## 10. Downstream Stages

This notebook stops after the main FairFace CLIP association analysis. The downstream research stages are kept in separate notebooks so each stage has a clean goal, method, and output directory.

- Stage 4 real-world labour-statistics comparison: `src/notebooks/stage4_realworld_distribution_comparison.ipynb`
- Stage 5 recommendation scenario simulation: `src/notebooks/stage5_recommendation_scenario_simulation.ipynb`

Run this notebook first because Stage 4 and Stage 5 load its saved tables and embedding files from `outputs/fairface_facet_broad/`.
